In [1]:
from main import run_bot
from demo import run_demo
import config

In [2]:
from config import CORRELATION_DIR, GENERAL_DIR, MIN_R2, OUTPUTS_DIR
from broker.connection import connect_ib
from broker.data import fetch_prices, fetch_prices_free
from broker.orders import calculate_position_size, execute_order, get_portfolio_value
from analysis.universe import fetch_market_caps, get_sp500_tickers
from analysis.correlations import compute_correlations, get_top_correlated_pairs, get_top_inverse_pairs
from analysis.model import predict_price
from analysis.signals import generate_signals
from reporting.charts import (plot_correlation_matrix, plot_market_cap_bars,
                               plot_prediction_analysis, plot_price_series)
from reporting.report import print_report, save_signals_csv

In [3]:
n_tickers = None    # int = top N by market cap | None = full S&P 500 | 'FALLBACK_TICKERS' = hardcoded top-20
mode = 'paper'      # demo | paper | live | signals
execute_trades=True

In [4]:
print("\nFetching S&P 500 universe...")
tickers, market_caps = get_sp500_tickers(n=n_tickers)


Fetching S&P 500 universe...
  ✓ 503 tickers fetched from Wikipedia
  Sorting 503 tickers by market cap via yfinance (this takes ~30s)...

Fetching market caps for 503 tickers via yfinance...
  ✓ Market caps retrieved: 503/503
  → Using all 503 S&P 500 tickers


In [5]:
prices_df = fetch_prices_free(tickers)
if prices_df.empty or len(prices_df.columns) < 5:
    print("✗ Insufficient data. Aborting.")

$TMUS: possibly delisted; no price data found  (period=1169d)

1 Failed download:
['TMUS']: possibly delisted; no price data found  (period=1169d)


  Tickers with data: 502/503
  ✓ A: 800 bars — last close $118.50
  ✓ AAPL: 800 bars — last close $288.14
  ✓ ABBV: 800 bars — last close $202.94
  ✓ ABNB: 800 bars — last close $139.82
  ✓ ABT: 800 bars — last close $87.64
  ✓ ACGL: 800 bars — last close $94.72
  ✓ ACN: 800 bars — last close $180.20
  ✓ ADBE: 800 bars — last close $257.06
  ✓ ADI: 800 bars — last close $408.69
  ✓ ADM: 800 bars — last close $77.84
  ✓ ADP: 800 bars — last close $214.10
  ✓ ADSK: 800 bars — last close $253.70
  ✓ AEE: 800 bars — last close $108.46
  ✓ AEP: 800 bars — last close $132.04
  ✓ AES: 800 bars — last close $14.31
  ✓ AFL: 800 bars — last close $113.16
  ✓ AIG: 800 bars — last close $76.49
  ✓ AIZ: 800 bars — last close $233.84
  ✓ AJG: 800 bars — last close $201.43
  ✓ AKAM: 800 bars — last close $114.75
  ✓ ALB: 800 bars — last close $204.24
  ✓ ALGN: 800 bars — last close $169.93
  ✓ ALL: 800 bars — last close $213.70
  ✓ ALLE: 800 bars — last close $136.84
  ✓ AMAT: 800 bars — last close $

In [6]:
print("\nCalculating correlations...")
corr_matrix, returns = compute_correlations(prices_df)
top_pairs     = get_top_correlated_pairs(corr_matrix, top_n=10)
inverse_pairs = get_top_inverse_pairs(corr_matrix, top_n=10)


Calculating correlations...


In [7]:
signals_df = generate_signals(prices_df, returns, corr_matrix)
print_report(signals_df, top_pairs, inverse_pairs)
save_signals_csv(signals_df)


Generating prediction signals (502 tickers)...
  [  1/502] ? A      $118.50 → $114.88  ret=-3.05%  R²=-0.64  [LOW_CONFIDENCE]
  [  2/502] ? AAPL   $288.14 → $nan  ret=N/A  R²=0.00  [INSUF_DATA]
  [  3/502] ? ABBV   $202.94 → $nan  ret=N/A  R²=0.00  [INSUF_DATA]
  [  4/502] ? ABNB   $139.82 → $140.61  ret=+0.57%  R²=-0.22  [LOW_CONFIDENCE]
  [  5/502] ? ABT    $87.64 → $nan  ret=N/A  R²=0.00  [INSUF_DATA]
  [  6/502] ? ACGL   $94.72 → $94.73  ret=+0.00%  R²=-0.29  [LOW_CONFIDENCE]
  [  7/502] ? ACN    $180.20 → $178.97  ret=-0.68%  R²=-0.26  [LOW_CONFIDENCE]
  [  8/502] ? ADBE   $257.06 → $266.19  ret=+3.55%  R²=-0.83  [LOW_CONFIDENCE]
  [  9/502] ? ADI    $408.69 → $414.37  ret=+1.39%  R²=-0.43  [LOW_CONFIDENCE]
  [ 10/502] ? ADM    $77.84 → $nan  ret=N/A  R²=0.00  [INSUF_DATA]
  [ 11/502] ? ADP    $214.10 → $217.41  ret=+1.54%  R²=-0.22  [LOW_CONFIDENCE]
  [ 12/502] ? ADSK   $253.70 → $261.89  ret=+3.23%  R²=-0.28  [LOW_CONFIDENCE]
  [ 13/502] ? AEE    $108.46 → $107.30  ret=-1.07%  

In [ ]:
save_plots = 1
if save_plots:
    plot_correlation_matrix(corr_matrix)

    # General/ — price series highlighted by market cap
    plot_price_series(prices_df, tickers, top_n=15, label='market cap',
                      save_path=GENERAL_DIR / 'price_series_market-cap.png')

    # General/ — price series highlighted by highest stock price
    tickers_by_price = sorted(
        prices_df.columns.tolist(),
        key=lambda t: prices_df[t].iloc[-1],
        reverse=True
    )
    plot_price_series(prices_df, tickers_by_price, top_n=15, label='stock price',
                      save_path=GENERAL_DIR / 'price_series_stock-price.png')

    # General/ — bar chart: top 15 vs bottom 15 by market cap
    plot_market_cap_bars(prices_df, tickers, market_caps=market_caps, top_n=15,
                         save_path=GENERAL_DIR / 'market_cap_bars.png')

    # Correlation_method/ — correlation heatmap is saved by default
    # Correlation_method/ — per-ticker prediction analysis
    top_signals = signals_df.head(5)
    if not top_signals.empty:
        print("\nGenerating analysis charts...")
    for _, row in top_signals.iterrows():
        ticker = row['ticker']
        pred_ret, r2, top5, corr_signs, y_actual, y_pred = predict_price(
            ticker, returns, corr_matrix
        )
        if y_actual is not None:
            plot_prediction_analysis(
                ticker, returns, prices_df, top5, corr_signs,
                y_actual, y_pred,
                save_path=CORRELATION_DIR / f'analysis_{ticker}.png'
            )

In [9]:
execute_trades = 0
if execute_trades:
    ib = connect_ib()
    try:
        portfolio_value = get_portfolio_value(ib)
        print(f"\nPlacing orders (portfolio: ${portfolio_value:,.0f})...")
        actionable = signals_df[signals_df['signal'].isin(['BUY', 'SELL'])]
        for _, row in actionable.iterrows():
            if row['model_r2'] < MIN_R2:
                continue
            strength = min(1.0, row['model_r2'])
            qty = calculate_position_size(portfolio_value, row['current_price'], strength)
            execute_order(ib, row['ticker'], row['signal'], qty)
    finally:
        ib.disconnect()
        print("\n✓ Disconnected from Interactive Brokers.")
else:
    print("\n  ℹ Simulation mode — no orders placed.")
    print("    To execute on paper trading: run_bot(execute_trades=True)")


  ℹ Simulation mode — no orders placed.
    To execute on paper trading: run_bot(execute_trades=True)
